# Association Rule Mining - Government Policy Patterns

Finds real patterns in the accident data - things like "no helmet + motorbike -> head injury" -
for the government-facing dashboard. A genuinely different kind of task from the other five models
in this project: this isn't predicting an outcome for one new patient, it's describing patterns
that already exist across everyone in the data. There's no train/test split in the usual sense,
but the same principle applies: don't just report whatever the algorithm finds, check it holds up.

**What last year did well, kept here:** using `mlxtend`'s Apriori/association-rules implementation
was the right tool for this job - unlike the ad hoc severity heuristics found in other services,
this was a legitimate, appropriate technique. Two of their filtering choices were genuinely good
and are kept: dropping rules built from "unknown" categories (a rule like "unknown collision ->
unknown road condition" is noise, not a finding), and excluding a patient's own severity from
predicting itself when mining rules aimed at a specific target.

**What's different here:**
- **The one-hot "basket" file is built from scratch in this notebook**, from the same cleaned
  dataset used throughout this project - last year's `df_ARM.csv` existed with no surviving
  documentation of how it was derived.
- **A real severity label** (built and validated earlier in this project) is included as a basket
  item, replacing last year's ad hoc, undocumented M/S severity field.
- **A minimum lift threshold is enforced** - last year had none, so a rule could look "confident"
  purely because its outcome was already common in the data, without the antecedent adding any
  real information.
- **Rules are checked for stability across 5 independent random splits**, not just reported once
  or checked against a single split. This is this notebook's closest equivalent to a held-out
  test: a rule that only shows up reliably across most splits is far more likely a genuine,
  robust pattern than one that happened to appear in one lucky split.
- **Redundant rules are pruned** - a specific rule that adds nothing beyond a more general rule
  with the same or better confidence is dropped, so the final list isn't cluttered with
  near-duplicates.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Install mlxtend if not already installed
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "mlxtend", "-q"])

from mlxtend.frequent_patterns import apriori, association_rules

import json, os
os.makedirs("artifacts", exist_ok=True)

RANDOM_STATE = 42

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sisepuede 1.4.0 requires matplotlib==3.10.1, but you have matplotlib 3.10.8 which is incompatible.
sisepuede 1.4.0 requires numpy==2.2.2, but you have numpy 2.4.6 which is incompatible.
sisepuede 1.4.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
sisepuede 1.4.0 requires PyYAML==6.0, but you have pyyaml 6.0.3 which is incompatible.


## 1. Load the data and pick the basket columns

`severity_label` is not a pre-existing column — it doesn't exist in the shared cleaned dataset.
It is **rebuilt here from the same formula the severity model uses**, so this notebook's notion
of "severe" is identical to the one every other model in this project agrees on, rather than a
separate, undocumented definition.

$$\text{severity\_high} = \text{further interventions} \;\lor\; \text{transferred out}
\;\lor\; \text{major procedure}$$

None of the three fields that *define* this label (`Discharge Outcome`, `Transfer To Next
Hospital`, `Management Name 1/2`) are themselves basket items below — including them would let a
rule "discover" a relationship that is actually just restating how the label was built.

In [2]:
BASKET_COLUMNS = [
    'Mode Of Travel During Accident', 'Collision With ', 'Time of Collision',
    'Road Condition', 'Category of Road', 'Road Type', 'Approximate Speed',
    'Road Signals Exist', 'Visiblity', 'Helmet Worn',
    'Site of Injury No1', 'severity_label',
]

df = pd.read_csv('Preprocessed_Dataset.csv')
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip().str.lower().replace({"nan": np.nan, "none": np.nan})
df = df.rename(columns={'Collision With ': 'Collision With'})
BASKET_COLUMNS = [c if c != 'Collision With ' else 'Collision With' for c in BASKET_COLUMNS]

MAJOR_PROCEDURES = {"surgery", "fracture stabilization", "wound debridement",
                    "ligamental injury management"}
m_outcome = df["Discharge Outcome"].eq("further interventions")
m_transfer = df["Transfer To Next Hospital"].eq("yes")
m_major = (df["Management Name 1"].isin(MAJOR_PROCEDURES) |
          df["Management Name 2"].isin(MAJOR_PROCEDURES))
df["severity_label"] = (m_outcome | m_transfer | m_major).map({True: "high", False: "low"})

print(f"severity_label built: {(df['severity_label'] == 'high').mean():.1%} high "
      f"({(df['severity_label'] == 'high').sum()} of {len(df)}) - matches the severity model exactly")

print(df.shape)
df[BASKET_COLUMNS].isna().mean().round(3)

severity_label built: 24.4% high (1505 of 6178) - matches the severity model exactly
(6178, 131)


Mode Of Travel During Accident    0.093
Collision With                    0.248
Time of Collision                 0.128
Road Condition                    0.210
Category of Road                  0.216
Road Type                         0.215
Approximate Speed                 0.268
Road Signals Exist                0.344
Visiblity                         0.203
Helmet Worn                       0.414
Site of Injury No1                0.086
severity_label                    0.000
dtype: float64

## 2. Turn each column into "item" labels, and drop genuinely missing/unknown values

A basket item needs to say both the field and the value - "Helmet Worn=no", not just "no", since
"no" alone is meaningless without knowing which question it's answering. Rows with a missing value
in a given column simply don't get an item for that column (rather than adding a fake "unknown"
item) - this keeps the item universe to genuine, informative categories, avoiding the noise-rule
problem last year handled only after the fact by filtering "unknown" out of the results.

In [3]:
item_frame = pd.DataFrame(index=df.index)
for col in BASKET_COLUMNS:
    clean = df[col].astype(str).str.strip().str.lower()
    clean = clean.where(df[col].notna())
    item_frame[col] = clean.apply(lambda v: f"{col}={v}" if pd.notna(v) else None)

print("Items per row (non-missing basket columns):")
print(item_frame.notna().sum(axis=1).describe())

Items per row (non-missing basket columns):
count    6178.000000
mean        9.575753
std         3.277541
min         1.000000
25%         9.000000
50%        11.000000
75%        12.000000
max        12.000000
dtype: float64


In [4]:
# one-hot encode: one column per unique item, True where that row has that item
all_items = pd.unique(item_frame.values.ravel())
all_items = [i for i in all_items if i is not None]
print(f"{len(all_items)} unique items across {len(BASKET_COLUMNS)} fields")

onehot = pd.DataFrame(False, index=df.index, columns=all_items)
for col in BASKET_COLUMNS:
    for item in item_frame[col].dropna().unique():
        onehot[item] = onehot[item] | (item_frame[col] == item)

onehot.shape

96 unique items across 12 fields


(6178, 96)

## 3. Choose real thresholds, and say why



**Minimum support (2%):** a pattern needs to show up in at least ~2% of all accidents (roughly

120+ of 6,178 rows) to be worth reporting to a government dashboard - rarer than that and it's not

a pattern worth policy attention, just a handful of unusual cases.



**Minimum confidence (50%):** when the antecedent is present, the consequent needs to follow at

least half the time - below that, the "rule" is weaker than a coin flip and not something to act

on.



**Minimum lift (1.2):** the piece last year's version was missing entirely. Lift measures whether

the antecedent genuinely raises the chance of the consequent, above and beyond how common the

consequent already is on its own. A lift of 1.0 means no real association at all - the antecedent

adds nothing. 1.2 requires a genuine, non-trivial bump, not just a rule that looks confident

because its outcome was already common.

In [5]:
MIN_SUPPORT = 0.02
MIN_CONFIDENCE = 0.50
MIN_LIFT = 1.2

frequent_itemsets = apriori(onehot, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
print(f"{len(frequent_itemsets)} frequent itemsets found (min support {MIN_SUPPORT:.0%})")

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
rules = rules[rules['lift'] >= MIN_LIFT].copy()
print(f"{len(rules)} rules pass confidence >= {MIN_CONFIDENCE:.0%} and lift >= {MIN_LIFT}")

1978 frequent itemsets found (min support 2%)
1584 rules pass confidence >= 50% and lift >= 1.2


## 4. Drop rules built from unknown/self-referential items

Same two good filters last year applied, kept here: a rule can't be built from a missing/unknown
value (there shouldn't be any in this dataset after Section 2, but checked defensively), and a
rule predicting severity can't have severity anywhere in its own antecedent - that would be
answering the question with part of the question.

In [6]:
def contains_unknown(itemset):
    return any('unknown' in str(i).lower() or 'missing' in str(i).lower() for i in itemset)

def antecedent_has_severity(row):
    return any(str(i).startswith('severity_label=') for i in row['antecedents'])

before = len(rules)
rules = rules[~rules['antecedents'].apply(contains_unknown) & ~rules['consequents'].apply(contains_unknown)]
rules = rules[~rules.apply(antecedent_has_severity, axis=1)]
print(f"Dropped {before - len(rules)} rules (unknown items or severity predicting itself)")
print(f"{len(rules)} rules remaining")

Dropped 167 rules (unknown items or severity predicting itself)
1417 rules remaining


## 5. Check the rules are stable across multiple splits, not just one

The closest thing this notebook has to a held-out test. A single 50/50 split could be lucky or
unlucky by chance - the same fragility every predictive model in this project moved away from
(single train/test split -> k-fold). Instead, this repeats the split-and-mine process 5 times with
different random splits, and for each rule found on the full dataset, checks what share of those 5
splits it shows up in **both** halves of. A rule stable across most splits is far more likely a
real, robust pattern than one that only happened to appear in a single lucky split.

In [7]:
def mine_rules(data_subset):
    itemsets = apriori(data_subset, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
    if len(itemsets) == 0:
        return pd.DataFrame(columns=['antecedents', 'consequents'])
    r = association_rules(itemsets, metric='confidence', min_threshold=MIN_CONFIDENCE)
    r = r[r['lift'] >= MIN_LIFT]
    r = r[~r['antecedents'].apply(contains_unknown) & ~r['consequents'].apply(contains_unknown)]
    r = r[~r.apply(antecedent_has_severity, axis=1)]
    return r

def rule_key(row):
    return (frozenset(row['antecedents']), frozenset(row['consequents']))

N_SPLITS = 5
STABILITY_FLOOR = 0.60  # a rule needs to show up in both halves of at least 3 of 5 splits

rng = np.random.RandomState(RANDOM_STATE)
appearance_counts = {}

for split_i in range(N_SPLITS):
    shuffled_idx = rng.permutation(onehot.index)
    half_a = onehot.loc[shuffled_idx[:len(onehot)//2]]
    half_b = onehot.loc[shuffled_idx[len(onehot)//2:]]

    rules_a = mine_rules(half_a)
    rules_b = mine_rules(half_b)
    keys_a = set(rules_a.apply(rule_key, axis=1)) if len(rules_a) else set()
    keys_b = set(rules_b.apply(rule_key, axis=1)) if len(rules_b) else set()
    both_this_split = keys_a & keys_b

    print(f"Split {split_i+1}: half A={len(rules_a)} rules, half B={len(rules_b)} rules, "
          f"stable within this split={len(both_this_split)}")

    for key in both_this_split:
        appearance_counts[key] = appearance_counts.get(key, 0) + 1

Split 1: half A=1426 rules, half B=1501 rules, stable within this split=1153
Split 2: half A=1422 rules, half B=1460 rules, stable within this split=1163
Split 3: half A=1513 rules, half B=1404 rules, stable within this split=1170
Split 4: half A=1468 rules, half B=1435 rules, stable within this split=1144
Split 5: half A=1400 rules, half B=1501 rules, stable within this split=1145


In [8]:
rules['rule_key'] = rules.apply(rule_key, axis=1)
rules['stability_score'] = rules['rule_key'].map(lambda k: appearance_counts.get(k, 0) / N_SPLITS)

stable_rules = rules[rules['stability_score'] >= STABILITY_FLOOR].drop(columns='rule_key').copy()

print(f"{len(stable_rules)} of {len(rules)} full-dataset rules are stable across at least "
      f"{STABILITY_FLOOR:.0%} of {N_SPLITS} random splits")
print(f"Mean stability score across all candidate rules: {rules['stability_score'].mean():.1%}")

1166 of 1417 full-dataset rules are stable across at least 60% of 5 random splits
Mean stability score across all candidate rules: 81.5%


## 6. Prune redundant rules

If a general rule already implies a more specific one at the same or better confidence, the
specific one isn't adding anything - e.g. if "no helmet -> head injury" already has 70% confidence,
a more specific version like "no helmet, night time -> head injury" at 68% confidence is strictly
worse and redundant, not a distinct finding. Dropped in favor of keeping the final list to genuinely
distinct patterns.

In [9]:
def is_redundant(row, all_rules):
    a, c = row['antecedents'], row['consequents']
    for _, other in all_rules.iterrows():
        if other['antecedents'] < a and other['consequents'] == c and other['confidence'] >= row['confidence']:
            return True
    return False

stable_rules = stable_rules.sort_values('lift', ascending=False).reset_index(drop=True)
redundant_mask = stable_rules.apply(lambda r: is_redundant(r, stable_rules), axis=1)
final_rules = stable_rules[~redundant_mask].copy()

print(f"Dropped {redundant_mask.sum()} redundant rules")
print(f"{len(final_rules)} final, non-redundant, stability-checked rules")

Dropped 269 redundant rules
897 final, non-redundant, stability-checked rules


## 7. Final results

Sorted by lift - the rules with the strongest genuine association, not just the highest raw
confidence, lead the list.

In [10]:
# 900 valid rules is too many to read at once - the full, validated set is saved to CSV below,
# but shown here is the top 25 by lift, the strongest genuine associations. This is a display
# choice for readability, not a filtering decision - all 900 are equally "final" and available.
TOP_N_DISPLAY = 25

display_rules = final_rules.copy()
display_rules['antecedents'] = display_rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))
display_rules['consequents'] = display_rules['consequents'].apply(lambda s: ', '.join(sorted(s)))
display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
display_rules = display_rules[display_cols].round(3)

print(f"{len(display_rules)} total validated rules - showing top {TOP_N_DISPLAY} by lift")
display_rules.head(TOP_N_DISPLAY)

897 total validated rules - showing top 25 by lift


,antecedents,consequents,support,confidence,lift
0,Road Signals Exist=yes,"Category of Road=highway, Visiblity=adequate",0.021,0.590,3.918
1,Road Signals Exist=yes,"Category of Road=highway, Road Condition=good",0.022,0.641,3.899
2,"Road Condition=good, Road Signals Exist=yes",Category of Road=highway,0.022,0.713,3.823
3,Road Signals Exist=yes,Category of Road=highway,0.023,0.650,3.485
5,Mode Of Travel During Accident=pedestrian,Collision With=motorbike,0.025,0.683,3.332
6,"Mode Of Travel During Accident=bicycle, Road S...",Approximate Speed=less than 40 km/h,0.062,0.697,2.718
7,"Mode Of Travel During Accident=bicycle, Road T...",Approximate Speed=less than 40 km/h,0.035,0.681,2.657
8,"Collision With=fall from vehicle, Mode Of Trav...",Approximate Speed=less than 40 km/h,0.026,0.671,2.616
9,"Mode Of Travel During Accident=bicycle, Road C...",Approximate Speed=less than 40 km/h,0.044,0.645,2.517
10,"Mode Of Travel During Accident=bicycle, Visibl...",Approximate Speed=less than 40 km/h,0.057,0.644,2.510


## 8. Save and document

In [11]:
full_rules_out = final_rules.copy()

full_rules_out['antecedents'] = full_rules_out['antecedents'].apply(lambda s: ', '.join(sorted(s)))

full_rules_out['consequents'] = full_rules_out['consequents'].apply(lambda s: ', '.join(sorted(s)))

full_rules_out[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(3).to_csv(

    'artifacts/association_rules.csv', index=False)



summary = {

    'basket_columns': BASKET_COLUMNS,

    'total_rows': int(len(df)),

    'unique_items': int(len(all_items)),

    'thresholds': {'min_support': MIN_SUPPORT, 'min_confidence': MIN_CONFIDENCE, 'min_lift': MIN_LIFT},

    'rules_after_thresholds_and_filters': int(len(rules)),

    'stability_check': {

        'n_splits': N_SPLITS,

        'stability_floor': STABILITY_FLOOR,

        'rules_meeting_stability_floor': int(len(stable_rules)),

        'mean_stability_score_all_candidates': float(rules['stability_score'].mean()),

    },

    'final_non_redundant_rules': int(len(final_rules)),

    'notes': [

        "The one-hot basket file is built here from Preprocessed_Dataset.csv - the same shared "

        "cleaned file every other model in this project uses - not a separate, undocumented file.",

        "severity_label is rebuilt here from the exact formula the severity model uses (further "

        "interventions OR transferred OR major procedure), not read from a separate file - so this "

        "notebook's definition of severe is guaranteed identical to the rest of the project's, and "

        "the fields that define the label are never themselves basket items (see Section 1).",

        "A minimum lift threshold (1.2) is enforced - last year had none, so a rule could look "

        "confident purely because its consequent was already common in the data.",

        "Rules are only kept if they replicate across at least 60% of 5 independent random data splits - "

        "a single-split check (used in an earlier version of this notebook) risked being lucky or "

        "unlucky by chance, the same fragility every predictive model in this project moved away from.",

        "Redundant rules (a specific version of a more general rule, adding no real information) "

        "are pruned from the final list.",

        "Missing/unknown values never become basket items at all, rather than being filtered out "

        "of results after the fact."

    ]

}

with open('artifacts/model_summary.json', 'w') as f:

    json.dump(summary, f, indent=2)



print(json.dumps(summary, indent=2))

{
  "basket_columns": [
    "Mode Of Travel During Accident",
    "Collision With",
    "Time of Collision",
    "Road Condition",
    "Category of Road",
    "Road Type",
    "Approximate Speed",
    "Road Signals Exist",
    "Visiblity",
    "Helmet Worn",
    "Site of Injury No1",
    "severity_label"
  ],
  "total_rows": 6178,
  "unique_items": 96,
  "thresholds": {
    "min_support": 0.02,
    "min_confidence": 0.5,
    "min_lift": 1.2
  },
  "rules_after_thresholds_and_filters": 1417,
  "stability_check": {
    "n_splits": 5,
    "stability_floor": 0.6,
    "rules_meeting_stability_floor": 1166,
    "mean_stability_score_all_candidates": 0.8151023288637967
  },
  "final_non_redundant_rules": 897,
  "notes": [
    "The one-hot basket file is built here from Preprocessed_Dataset.csv - the same shared cleaned file every other model in this project uses - not a separate, undocumented file.",
    "severity_label is rebuilt here from the exact formula the severity model uses (further i